# EDA — Dataset sintético embarazo (solo tablas)

Exploración tabular del dataset generado. Sin gráficos.

**Fuentes:**
- `../data/datos_embarazo_sintetico.csv`
- `../data/metadatos_ground_truth.csv`
- `../data/missingness_log.csv`

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")

df = pd.read_csv(DATA_DIR / "datos_embarazo_sintetico.csv")
meta = pd.read_csv(DATA_DIR / "metadatos_ground_truth.csv")
log_miss = pd.read_csv(DATA_DIR / "missingness_log.csv")

df_full = df.merge(meta, on="paciente_id", how="left")

NUMERIC = df.select_dtypes(include=[np.number]).columns.tolist()
CATEGORIC = df.select_dtypes(include=["object"]).columns.tolist()
BINARY = [c for c in NUMERIC if set(df[c].dropna().unique()).issubset({0, 0.0, 1, 1.0})]
CONTINUOUS = [c for c in NUMERIC if c not in BINARY and c != "paciente_id"]

C:\Users\MAXIMILIANO\AppData\Local\Temp\ipykernel_12348\819094590.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  CATEGORIC = df.select_dtypes(include=["object"]).columns.tolist()


## 1. Dimensiones y tipos

In [4]:
resumen_general = pd.DataFrame({
    "metrica": ["Filas", "Columnas features", "Variables numéricas", "Variables categóricas", "Duplicados paciente_id"],
    "valor": [len(df), len(df.columns) - 1, len(NUMERIC), len(CATEGORIC), df["paciente_id"].duplicated().sum()],
})
display(resumen_general)

display(pd.DataFrame({
    "columna": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "no_nulos": df.notna().sum().values,
    "pct_completo": (100 * df.notna().mean()).round(2).values,
}))

,metrica,valor
0,Filas,2000
1,Columnas features,20
2,Variables numéricas,19
3,Variables categóricas,2
4,Duplicados paciente_id,0


,columna,dtype,no_nulos,pct_completo
0,paciente_id,int64,2000,100.00
1,edad_anios,float64,1942,97.10
2,imc_pregestacional,float64,1947,97.35
3,semanas_gestacion,float64,1928,96.40
4,trimestre,int64,2000,100.00
5,num_embarazos_previos,float64,1927,96.35
6,num_partos_previos,float64,1889,94.45
7,embarazo_multiple,int64,2000,100.00
8,tabaquismo_activo,float64,1921,96.05
9,diabetes_previa,int64,2000,100.00


## 2. Muestra de registros

In [5]:
display(df_full.head(10))
display(df_full.tail(5))

,paciente_id,edad_anios,imc_pregestacional,semanas_gestacion,trimestre,num_embarazos_previos,num_partos_previos,embarazo_multiple,tabaquismo_activo,diabetes_previa,hipertension_cronica,peso_kg,talla_cm,ganancia_peso_kg,presion_sistolica,presion_diastolica,frecuencia_cardiaca,nivel_educacion,area_residencia,suplemento_acido_folico,suplemento_hierro,cluster_verdadero,es_outlier,tipo_outlier,mecanismo_nulo
0,1,33.60,17.67,26.00,2,0.00,0.00,0,0.00,0,0,40.50,155.50,-1.70,123.00,70.00,88.00,primaria,urbana,0.00,0.00,3,0,ninguno,{}
1,2,33.80,34.17,29.20,3,0.00,0.00,0,0.00,1,1,100.00,167.00,9.00,115.00,85.00,72.00,superior,urbana,1.00,1.00,1,0,ninguno,{}
2,3,41.30,34.31,26.70,2,3.00,3.00,0,0.00,0,0,99.80,NaN,2.30,99.00,77.00,80.00,secundaria,urbana,0.00,0.00,1,0,ninguno,"{""talla_cm"": ""MCAR""}"
3,4,36.10,26.13,30.70,3,0.00,0.00,0,0.00,0,1,75.00,165.30,3.30,134.00,89.00,93.00,primaria,urbana,1.00,1.00,2,0,ninguno,{}
4,5,33.80,28.33,38.10,3,0.00,0.00,0,0.00,0,0,87.30,164.00,8.80,113.00,76.00,82.00,primaria,rural,0.00,1.00,1,0,ninguno,{}
5,6,27.70,35.17,36.00,3,1.00,1.00,0,0.00,0,0,-23.60,165.30,74.10,92.00,67.00,92.00,superior,urbana,1.00,1.00,1,1,combinacion_imposible,{}
6,7,38.80,27.46,34.00,3,0.00,NaN,0,0.00,0,1,78.50,167.70,0.00,117.00,89.00,87.00,secundaria,urbana,0.00,0.00,2,0,ninguno,"{""num_partos_previos"": ""MAR""}"
7,8,41.00,15.46,32.50,3,0.00,0.00,0,0.00,0,1,46.10,164.40,2.90,94.00,66.00,90.00,primaria,rural,1.00,1.00,4,0,ninguno,{}
8,9,17.00,18.46,35.70,3,0.00,0.00,1,0.00,0,1,54.60,166.30,2.00,188.00,107.00,78.00,secundaria,urbana,1.00,1.00,2,1,patologia_rara,{}
9,10,20.00,17.75,32.20,3,2.00,2.00,0,0.00,0,0,40.00,150.20,-2.00,102.00,58.00,77.00,secundaria,rural,1.00,1.00,3,0,ninguno,{}


,paciente_id,edad_anios,imc_pregestacional,semanas_gestacion,trimestre,num_embarazos_previos,num_partos_previos,embarazo_multiple,tabaquismo_activo,diabetes_previa,hipertension_cronica,peso_kg,talla_cm,ganancia_peso_kg,presion_sistolica,presion_diastolica,frecuencia_cardiaca,nivel_educacion,area_residencia,suplemento_acido_folico,suplemento_hierro,cluster_verdadero,es_outlier,tipo_outlier,mecanismo_nulo
1995,1996,19.90,17.32,37.00,3,0.00,0.00,0,0.00,0,0,40.00,158.20,-1.40,95.00,74.00,80.00,primaria,urbana,1.00,0.00,3,0,ninguno,{}
1996,1997,25.20,22.84,29.30,3,5.00,5.00,0,0.00,0,0,60.90,165.40,-1.80,105.00,65.00,72.00,superior,urbana,1.00,1.00,0,0,ninguno,{}
1997,1998,28.20,21.88,18.40,2,0.00,0.00,0,0.00,0,0,53.90,156.70,-1.50,124.00,74.00,76.00,secundaria,urbana,1.00,1.00,0,0,ninguno,{}
1998,1999,33.40,22.77,28.60,3,0.00,0.00,0,0.00,0,0,58.20,160.70,1.50,100.00,NaN,70.00,primaria,urbana,1.00,1.00,0,0,ninguno,"{""presion_diastolica"": ""MAR""}"
1999,2000,24.10,17.60,75.00,2,0.00,0.00,1,0.00,0,0,42.20,NaN,NaN,123.00,51.00,75.00,secundaria,rural,1.00,1.00,3,1,medicion,"{""talla_cm"": ""MCAR"", ""ganancia_peso_kg"": ""MCAR""}"


## 3. Estadísticos descriptivos — variables continuas

In [6]:
desc = df[CONTINUOUS].describe().T
desc["IQR"] = desc["75%"] - desc["25%"]
desc["CV"] = (desc["std"] / desc["mean"]).replace([np.inf, -np.inf], np.nan)
display(desc.round(2))

,count,mean,std,min,25%,50%,75%,max,IQR,CV
edad_anios,"1,942.00",30.42,12.39,-17.00,25.80,29.50,33.90,340.00,8.10,0.41
imc_pregestacional,"1,947.00",25.07,6.49,-27.20,21.04,24.11,28.96,119.75,7.91,0.26
semanas_gestacion,"1,928.00",29.62,7.80,-29.90,24.50,30.10,34.92,94.80,10.42,0.26
trimestre,"2,000.00",2.63,0.52,1.00,2.00,3.00,3.00,3.00,1.00,0.20
num_embarazos_previos,"1,927.00",1.09,1.69,0.00,0.00,0.00,2.00,5.00,2.00,1.56
num_partos_previos,"1,889.00",0.58,1.13,0.00,0.00,0.00,1.00,5.00,1.00,1.95
peso_kg,"1,925.00",68.34,33.13,-23.60,54.50,64.90,78.80,904.00,24.30,0.48
talla_cm,"1,928.00",164.88,64.83,-136.50,157.40,161.90,166.52,"1,612.00",9.12,0.39
ganancia_peso_kg,"1,935.00",1.95,3.73,-26.60,0.10,1.60,3.30,74.10,3.20,1.91
presion_sistolica,"1,937.00",111.27,18.73,-104.00,100.00,111.00,121.00,243.00,21.00,0.17


## 4. Percentiles detallados

In [7]:
percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
pct_table = df[CONTINUOUS].quantile(percentiles).T
pct_table.columns = [f"p{int(p*100)}" for p in percentiles]
display(pct_table.round(2))

,p1,p5,p10,p25,p50,p75,p90,p95,p99
edad_anios,15.20,19.80,22.20,25.80,29.50,33.90,38.50,41.50,47.16
imc_pregestacional,15.56,16.71,17.57,21.04,24.11,28.96,33.55,35.18,38.39
semanas_gestacion,11.03,16.67,19.74,24.50,30.10,34.92,39.70,41.90,42.00
trimestre,1.00,2.00,2.00,2.00,3.00,3.00,3.00,3.00,3.00
num_embarazos_previos,0.00,0.00,0.00,0.00,0.00,2.00,4.00,5.00,5.00
num_partos_previos,0.00,0.00,0.00,0.00,0.00,1.00,2.00,3.00,4.00
peso_kg,40.00,40.00,42.70,54.50,64.90,78.80,95.88,102.40,115.18
talla_cm,148.10,151.60,153.50,157.40,161.90,166.52,170.73,173.60,179.30
ganancia_peso_kg,-2.00,-2.00,-1.50,0.10,1.60,3.30,5.60,7.20,9.23
presion_sistolica,82.00,88.00,92.00,100.00,111.00,121.00,132.00,139.20,156.64


## 5. Variables binarias — frecuencias

In [8]:
filas_bin = []
for col in BINARY:
    vc = df[col].value_counts(dropna=False)
    for val, cnt in vc.items():
        filas_bin.append({
            "variable": col,
            "valor": val,
            "conteo": cnt,
            "pct": round(100 * cnt / len(df), 2),
        })
display(pd.DataFrame(filas_bin).sort_values(["variable", "valor"]))

,variable,valor,conteo,pct
5,diabetes_previa,0.00,1753,87.65
6,diabetes_previa,1.00,247,12.35
0,embarazo_multiple,0.00,1886,94.30
1,embarazo_multiple,1.00,114,5.70
7,hipertension_cronica,0.00,1699,84.95
8,hipertension_cronica,1.00,301,15.05
10,suplemento_acido_folico,0.00,263,13.15
9,suplemento_acido_folico,1.00,1711,85.55
11,suplemento_acido_folico,NaN,26,1.30
13,suplemento_hierro,0.00,347,17.35


## 6. Variables categóricas — frecuencias

In [9]:
filas_cat = []
for col in CATEGORIC:
    vc = df[col].value_counts(dropna=False)
    for val, cnt in vc.items():
        filas_cat.append({
            "variable": col,
            "categoria": val,
            "conteo": cnt,
            "pct": round(100 * cnt / len(df), 2),
        })
display(pd.DataFrame(filas_cat).sort_values(["variable", "conteo"], ascending=[True, False]))

,variable,categoria,conteo,pct
3,area_residencia,urbana,1279,63.95
4,area_residencia,rural,721,36.05
0,nivel_educacion,secundaria,714,35.70
1,nivel_educacion,superior,662,33.10
2,nivel_educacion,primaria,624,31.20


## 7. Valores faltantes por columna

In [10]:
missing = pd.DataFrame({
    "variable": df.columns,
    "nulos": df.isna().sum().values,
    "pct_nulos": (100 * df.isna().mean()).round(2).values,
    "completos": df.notna().sum().values,
}).sort_values("nulos", ascending=False)
display(missing)

filas_con_algun_nulo = df.isna().any(axis=1).sum()
display(pd.DataFrame({
    "metrica": ["Filas con al menos un nulo", "Filas completas"],
    "conteo": [filas_con_algun_nulo, len(df) - filas_con_algun_nulo],
    "pct": [round(100 * filas_con_algun_nulo / len(df), 2), round(100 * (len(df) - filas_con_algun_nulo) / len(df), 2)],
}))

,variable,nulos,pct_nulos,completos
15,presion_diastolica,359,17.95,1641
6,num_partos_previos,111,5.55,1889
8,tabaquismo_activo,79,3.95,1921
11,peso_kg,75,3.75,1925
5,num_embarazos_previos,73,3.65,1927
3,semanas_gestacion,72,3.60,1928
12,talla_cm,72,3.60,1928
13,ganancia_peso_kg,65,3.25,1935
14,presion_sistolica,63,3.15,1937
1,edad_anios,58,2.90,1942


,metrica,conteo,pct
0,Filas con al menos un nulo,894,44.70
1,Filas completas,1106,55.30


## 8. Missingness por mecanismo (ground truth)

In [11]:
display(log_miss.groupby("mecanismo").size().reset_index(name="celdas_afectadas"))
display(
    log_miss.groupby(["variable", "mecanismo"])
    .size()
    .reset_index(name="conteo")
    .sort_values("conteo", ascending=False)
)

,mecanismo,celdas_afectadas
0,MAR,420
1,MCAR,602
2,MNAR,136


,variable,mecanismo,conteo
9,presion_diastolica,MAR,305
6,num_partos_previos,MAR,111
16,tabaquismo_activo,MNAR,79
5,num_embarazos_previos,MCAR,73
13,semanas_gestacion,MCAR,72
17,talla_cm,MCAR,72
3,ganancia_peso_kg,MCAR,62
0,edad_anios,MCAR,58
7,peso_kg,MCAR,58
10,presion_diastolica,MCAR,54


## 9. Outliers inyectados (ground truth)

In [12]:
display(meta.groupby("es_outlier").size().reset_index(name="pacientes"))
display(
    meta[meta["es_outlier"] == 1]
    .groupby("tipo_outlier")
    .size()
    .reset_index(name="conteo")
    .sort_values("conteo", ascending=False)
)
display(
    meta.groupby(["cluster_verdadero", "es_outlier"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "sin_outlier", 1: "con_outlier"})
)

,es_outlier,pacientes
0,0,1903
1,1,97


,tipo_outlier,conteo
1,medicion,45
4,registro,24
0,combinacion_imposible,18
2,patologia_rara,7
3,posicion,3


es_outlier,sin_outlier,con_outlier
cluster_verdadero,,
0,832,43
1,361,19
2,210,11
3,304,15
4,139,7
5,57,2


## 10. Distribución de clusters verdaderos

In [13]:
CLUSTER_LABELS = {
    0: "C0 — Bajo riesgo",
    1: "C1 — Riesgo metabólico",
    2: "C2 — Riesgo hipertensivo",
    3: "C3 — Bajo peso nutricional",
    4: "C4 — Alto riesgo obstétrico",
    5: "C5 — Residual",
}

dist_cluster = meta["cluster_verdadero"].value_counts().sort_index().reset_index()
dist_cluster.columns = ["cluster", "conteo"]
dist_cluster["pct"] = (100 * dist_cluster["conteo"] / len(meta)).round(2)
dist_cluster["perfil"] = dist_cluster["cluster"].map(CLUSTER_LABELS)
display(dist_cluster)

,cluster,conteo,pct,perfil
0,0,875,43.75,C0 — Bajo riesgo
1,1,380,19.00,C1 — Riesgo metabólico
2,2,221,11.05,C2 — Riesgo hipertensivo
3,3,319,15.95,C3 — Bajo peso nutricional
4,4,146,7.30,C4 — Alto riesgo obstétrico
5,5,59,2.95,C5 — Residual


## 11. Estadísticos por cluster verdadero

In [14]:
vars_perfil = [
    "edad_anios", "imc_pregestacional", "semanas_gestacion",
    "ganancia_peso_kg", "presion_sistolica", "presion_diastolica", "frecuencia_cardiaca",
]

por_cluster = (
    df_full.groupby("cluster_verdadero")[vars_perfil]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .round(2)
)
display(por_cluster)

edad_anios                                   \
                       count  mean   std    min median    max   
cluster_verdadero                                               
0                        851 29.07 16.10 -17.00  28.20 340.00   
1                        369 32.04  6.36   8.00  32.20  99.00   
2                        212 36.76  8.68  17.00  36.35 113.00   
3                        310 24.22  5.24   3.00  24.35  37.80   
4                        141 38.75  4.30  29.00  38.70  49.00   
5                         59 29.67  7.10   7.00  29.50  43.90   

                  imc_pregestacional                                  \
                               count  mean  std    min median    max   
cluster_verdadero                                                      
0                                852 23.35 3.32 -14.46  23.10  72.71   
1                                369 32.58 4.32 -27.20  32.74  51.09   
2                                215 28.09 8.20  18.38  27.24 119.75   
3                                312 17.67 2.53  14.66  17.30  37.21   
4                                141 27.33 6.04  14.00  28.00  56.66   
5                                 58 25.84 4.63  15.87  26.36  35.15   

                  semanas_gestacion                                 \
                              count  mean  std    min median   max   
cluster_verdadero                                                    
0                               845 30.14 7.99   6.00  30.20 94.80   
1                               369 27.74 7.92 -29.90  28.20 42.00   
2                               213 31.51 6.32  -6.10  31.90 42.00   
3                               302 28.34 8.19   6.00  28.30 75.00   
4                               141 32.99 4.99  13.90  33.50 42.00   
5                                58 25.61 7.18  12.60  24.05 42.00   

                  ganancia_peso_kg                                 \
                             count  mean  std    min median   max   
cluster_verdadero                                                   
0                              850  1.66 3.07 -26.60   1.60 62.80   
1                              366  5.91 4.53  -3.20   5.70 74.10   
2                              217  1.02 2.04 -20.10   1.10  5.10   
3                              303 -1.23 0.98  -8.60  -1.50  1.70   
4                              142  2.06 2.64  -4.00   1.95 26.90   
5                               57  1.04 1.60  -4.50   0.80  4.80   

                  presion_sistolica                                     \
                              count   mean   std     min median    max   
cluster_verdadero                                                        
0                               847 105.43 14.29    1.00 105.00 243.00   
1                               369 115.31 17.04 -104.00 116.00 186.00   
2                               214 133.64 18.63  -47.00 134.00 188.00   
3                               311 102.89 15.65  -47.00 103.00 178.00   
4                               139 120.97 16.25   91.00 121.00 205.00   
5                                57 109.86 25.50  -37.00 112.00 148.00   

                  presion_diastolica                                   \
                               count  mean   std    min median    max   
cluster_verdadero                                                       
0                                694 68.32  6.61 -23.00  68.00 112.00   
1                                326 78.61  8.46 -18.00  78.00 111.00   
2                                208 96.31  8.58  71.00  96.00 117.00   
3                                243 66.17  7.97  51.00  65.00 121.00   
4                                120 83.56  8.24  60.00  84.00 104.00   
5                                 50 75.20 10.53  49.00  74.50  99.00   

                  frecuencia_cardiaca                                   
                                count  mean   std    min median    max  
cluster_verdadero                                          

## 12. Tablas cruzadas

In [15]:
display(pd.crosstab(df_full["cluster_verdadero"], df_full["trimestre"], margins=True))
display(pd.crosstab(df_full["cluster_verdadero"], df_full["diabetes_previa"], margins=True))
display(pd.crosstab(df_full["cluster_verdadero"], df_full["hipertension_cronica"], margins=True))
display(pd.crosstab(df_full["nivel_educacion"], df_full["area_residencia"], margins=True))

trimestre,1,2,3,All
cluster_verdadero,,,,
0,15,274,586,875
1,9,161,210,380
2,0,54,167,221
3,9,132,178,319
4,0,15,131,146
5,2,33,24,59
All,35,669,1296,2000


diabetes_previa,0,1,All
cluster_verdadero,,,
0,858,17,875
1,226,154,380
2,192,29,221
3,312,7,319
4,109,37,146
5,56,3,59
All,1753,247,2000


hipertension_cronica,0,1,All
cluster_verdadero,,,
0,838,37,875
1,321,59,380
2,93,128,221
3,302,17,319
4,93,53,146
5,52,7,59
All,1699,301,2000


area_residencia,rural,urbana,All
nivel_educacion,,,
primaria,207,417,624
secundaria,269,445,714
superior,245,417,662
All,721,1279,2000


## 13. Matriz de correlación (Pearson)

In [16]:
corr = df[CONTINUOUS].corr(method="pearson").round(3)
display(corr)

pares = []
cols = corr.columns.tolist()
for i, a in enumerate(cols):
    for b in cols[i + 1:]:
        pares.append({"var_a": a, "var_b": b, "r": corr.loc[a, b]})
pares_df = pd.DataFrame(pares).sort_values("r", key=abs, ascending=False)
display(pares_df.head(15))

,edad_anios,imc_pregestacional,semanas_gestacion,trimestre,num_embarazos_previos,num_partos_previos,peso_kg,talla_cm,ganancia_peso_kg,presion_sistolica,presion_diastolica,frecuencia_cardiaca
edad_anios,1.00,0.20,0.05,0.08,0.03,0.01,0.12,-0.10,0.08,0.06,0.27,0.05
imc_pregestacional,0.20,1.00,-0.01,0.02,0.06,0.04,0.49,-0.02,0.44,0.23,0.39,0.07
semanas_gestacion,0.05,-0.01,1.00,0.77,0.01,0.00,-0.01,-0.03,0.06,0.06,0.04,0.03
trimestre,0.08,0.02,0.77,1.00,-0.00,-0.01,-0.01,-0.03,0.07,0.05,0.07,0.02
num_embarazos_previos,0.03,0.06,0.01,-0.00,1.00,0.75,0.07,-0.00,0.07,-0.01,-0.01,0.00
num_partos_previos,0.01,0.04,0.00,-0.01,0.75,1.00,0.05,0.03,0.04,-0.01,0.02,-0.01
peso_kg,0.12,0.49,-0.01,-0.01,0.07,0.05,1.00,0.00,0.26,0.15,0.20,0.02
talla_cm,-0.10,-0.02,-0.03,-0.03,-0.00,0.03,0.00,1.00,-0.01,0.01,-0.01,0.04
ganancia_peso_kg,0.08,0.44,0.06,0.07,0.07,0.04,0.26,-0.01,1.00,0.11,0.09,0.03
presion_sistolica,0.06,0.23,0.06,0.05,-0.01,-0.01,0.15,0.01,0.11,1.00,0.69,0.08


,var_a,var_b,r
21,semanas_gestacion,trimestre,0.77
38,num_embarazos_previos,num_partos_previos,0.75
63,presion_sistolica,presion_diastolica,0.69
15,imc_pregestacional,peso_kg,0.49
17,imc_pregestacional,ganancia_peso_kg,0.44
19,imc_pregestacional,presion_diastolica,0.39
9,edad_anios,presion_diastolica,0.27
52,peso_kg,ganancia_peso_kg,0.26
18,imc_pregestacional,presion_sistolica,0.23
65,presion_diastolica,frecuencia_cardiaca,0.21


## 14. Detección de atípicos univariados (regla IQR)

In [17]:
iqr_rows = []
for col in CONTINUOUS:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    iqr_rows.append({
        "variable": col,
        "Q1": round(q1, 2),
        "Q3": round(q3, 2),
        "IQR": round(iqr, 2),
        "limite_inf": round(lo, 2),
        "limite_sup": round(hi, 2),
        "n_atipicos_iqr": int(n_out),
        "pct_atipicos": round(100 * n_out / len(df), 2),
    })
display(pd.DataFrame(iqr_rows).sort_values("n_atipicos_iqr", ascending=False))

,variable,Q1,Q3,IQR,limite_inf,limite_sup,n_atipicos_iqr,pct_atipicos
5,num_partos_previos,0.00,1.00,1.00,-1.50,2.50,177,8.85
10,presion_diastolica,67.00,81.00,14.00,46.00,102.00,57,2.85
8,ganancia_peso_kg,0.10,3.30,3.20,-4.70,8.10,56,2.80
11,frecuencia_cardiaca,77.00,87.00,10.00,62.00,102.00,38,1.90
9,presion_sistolica,100.00,121.00,21.00,68.50,152.50,37,1.85
0,edad_anios,25.80,33.90,8.10,13.65,46.05,32,1.60
6,peso_kg,54.50,78.80,24.30,18.05,115.25,24,1.20
7,talla_cm,157.40,166.52,9.12,143.71,180.21,23,1.15
2,semanas_gestacion,24.50,34.92,10.42,8.86,50.56,11,0.55
1,imc_pregestacional,21.04,28.96,7.91,9.17,40.83,9,0.45


## 15. Perfil por trimestre gestacional

In [18]:
por_trimestre = (
    df.groupby("trimestre")[vars_perfil]
    .agg(["count", "mean", "std"])
    .round(2)
)
display(por_trimestre)

edad_anios             imc_pregestacional             \
               count  mean   std              count  mean  std   
trimestre                                                        
1                 34 27.00  5.53                 34 23.66 5.85   
2                649 29.21  6.15                656 25.06 6.21   
3               1259 31.14 14.66               1257 25.12 6.65   

          semanas_gestacion            ganancia_peso_kg            \
                      count  mean  std            count mean  std   
trimestre                                                           
1                        33 10.87 2.03               32 1.15 2.48   
2                       645 22.26 4.78              648 1.61 3.14   
3                      1250 33.92 5.14             1255 2.15 4.01   

          presion_sistolica              presion_diastolica              \
                      count   mean   std              count  mean   std   
trimestre                                                                 
1                        35 106.14 10.49                 25 70.36  6.68   
2                       652 110.33 17.49                556 73.98 11.01   
3                      1250 111.90 19.49               1060 75.52 13.23   

          frecuencia_cardiaca              
                        count  mean   std  
trimestre                                  
1                          34 83.71  7.88  
2                         654 82.17  9.62  
3                        1261 82.81 10.01

## 16. Coherencia clínica básica (reglas duras)

In [19]:
validaciones = []

pa_invalida = df.dropna(subset=["presion_sistolica", "presion_diastolica"])
pa_invalida = pa_invalida[pa_invalida["presion_sistolica"] <= pa_invalida["presion_diastolica"] + 14]
validaciones.append({"regla": "PA sistólica > diastólica + 14", "violaciones": len(pa_invalida)})

paridad = df.dropna(subset=["num_embarazos_previos", "num_partos_previos"])
paridad_bad = paridad[paridad["num_partos_previos"] > paridad["num_embarazos_previos"]]
validaciones.append({"regla": "partos <= embarazos previos", "violaciones": len(paridad_bad)})

trim_bad = df[df["trimestre"] != df["semanas_gestacion"].apply(
    lambda s: 1 if s <= 13 else (2 if s <= 27 else 3) if pd.notna(s) else np.nan
)]
validaciones.append({"regla": "trimestre coherente con semanas", "violaciones": len(trim_bad)})

display(pd.DataFrame(validaciones))

,regla,violaciones
0,PA sistólica > diastólica + 14,3
1,partos <= embarazos previos,0
2,trimestre coherente con semanas,87
